In [1]:
import numpy as np 
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import tiktoken
import importlib
from gpt_download import download_and_load_gpt2
# from gpt_model import DummyGPTModel


/home/dharmsingh/miniconda3/envs/ml/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
I0000 00:00:1783274919.280970   58388 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783274919.407259   58388 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783274922.073363   58388 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from dif

In [2]:
import new_gpt

In [3]:
print(new_gpt.__file__)
importlib.reload(new_gpt)

/home/dharmsingh/data/llm-from-scratch/new_gpt.py


<module 'new_gpt' from '/home/dharmsingh/data/llm-from-scratch/new_gpt.py'>

In [4]:
load_weights_into_gpt = new_gpt.load_weights_into_gpt
DummyGPTModel = new_gpt.DummyGPTModel

In [5]:
model_configs = {
"gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
"gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
"gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
"gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
GPT_CONFIG_124M = {
"vocab_size": 50257,
"context_length": 256,
"emb_dim": 768,
"n_heads": 12,
"n_layers": 12,
"drop_rate": 0.1,
"qkv_bias": False
}


In [6]:
# import urllib.request
# import zipfile
# import os
# from pathlib import Path
# url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
# zip_path = "sms_spam_collection.zip"
# extracted_path = "sms_spam_collection"
# data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"
# def download_and_unzip_spam_data(
#     url, zip_path, extracted_path, data_file_path):
#     if data_file_path.exists():
#         print(f"{data_file_path} already exists. Skipping download "
# "and extraction."
# )
#         return
#     with urllib.request.urlopen(url) as response:
#         with open(zip_path, "wb") as out_file:
#             out_file.write(response.read())
#     with zipfile.ZipFile(zip_path, "r") as zip_ref:
#         zip_ref.extractall(extracted_path)

#     original_file_path = Path(extracted_path) / "SMSSpamCollection"
#     os.rename(original_file_path, data_file_path)
#     print(f"File downloaded and saved as {data_file_path}")

# download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

In [7]:

df = pd.read_csv(
"./sms_spam_collection/SMSSpamCollection.tsv", sep="\t", header=None, names=["Label", "Text"]
)
df.shape

(5572, 2)

In [8]:
df["Label"].value_counts()

Label
ham     4825
spam     747
Name: count, dtype: int64

In [9]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(
        num_spam , random_state = 123
    )
    balanced_dataset = pd.concat([
        ham_subset,df[df["Label"] == "spam"]
    ])
    return balanced_dataset

balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [10]:
balanced_df["Label"] = balanced_df["Label"].map({"ham":0 , "spam" :1})

In [11]:
balanced_df

,Label,Text
4307,0,Awww dat is sweet! We can think of something t...
4138,0,Just got to &lt;#&gt;
4831,0,"The word ""Checkmate"" in chess comes from the P..."
4461,0,This is wishing you a great day. Moji told me ...
5440,0,Thank you. do you generally date the brothas?
...,...,...
5537,1,Want explicit SEX in 30 secs? Ring 02073162414...
5540,1,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,1,Had your contract mobile 11 Mnths? Latest Moto...
5566,1,REMINDER FROM O2: To get 2.50 pounds free call...


In [12]:
def random_split(df, train_frac, validation_frac):
    df = df.sample(
        frac=1,random_state= 123
    )
    train_end = int(len(df) * train_frac)
    validation_end = train_end +  int(len(df) * validation_frac)
    
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]
    
    return train_df,validation_df,test_df
train_df, validation_df, test_df = random_split(
balanced_df, 0.7, 0.1)
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

In [13]:
class SpamDataset(Dataset):
    def __init__(self ,csv_file, tokenizer , max_length =None , pad_token_id = 50256):
        super().__init__()
        self.data = pd.read_csv(csv_file)
        
        self.encoded_texts = [tokenizer.encode(text) for text in self.data["Text"]]
        
        if max_length is None:
            self.max_length = self._longest_encoded_length()
            
        else:
            self.max_length = max_length
            
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
                
            ]
        self.encoded_texts = [
                encoded_text + [pad_token_id] *
                (self.max_length - len(encoded_text))
                for encoded_text in self.encoded_texts
                ]
        
    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
        torch.tensor(encoded, dtype=torch.long),
        torch.tensor(label, dtype=torch.long)
        )
        
    def __len__(self):
        return len(self.data)
    
    def _longest_encoded_length(self):
        max_length =0
        for encoded_text  in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length
            

In [14]:
tokenizer = tiktoken.get_encoding("gpt2")
train_dataset =SpamDataset(csv_file="./train.csv" ,tokenizer=tokenizer , max_length=None )
val_dataset =SpamDataset(csv_file="./validation.csv" ,tokenizer=tokenizer , max_length=None )
test_dataset =SpamDataset(csv_file="./test.csv" ,tokenizer=tokenizer , max_length=None )



In [15]:
num_workers = 0
batch_size = 8
torch.manual_seed(123)
train_loader = DataLoader(train_dataset,batch_size,shuffle=True,num_workers=num_workers,drop_last=True)
val_loader = DataLoader(val_dataset,batch_size,shuffle=True,num_workers=num_workers,drop_last=True)
test_loader = DataLoader(test_dataset,batch_size,shuffle=True,num_workers=num_workers,drop_last=True)


In [16]:
for input_batch, target_batch in train_loader:
    pass
print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

Input batch dimensions: torch.Size([8, 120])
Label batch dimensions torch.Size([8])


In [17]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"
BASE_CONFIG = {

"vocab_size": 50257,

"context_length": 1024,
"drop_rate": 0.0,

"qkv_bias": True

}
model_configs = {
"gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
"gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
"gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
"gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [18]:
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
model_size=model_size, models_dir="gpt2"
)
model = DummyGPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe


DummyGPTModel(
  (token_embed): Embedding(50257, 768)
  (pos_embed): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_block): Sequential(
    (0): DummyTransformerBlock(
      (attention): MultiheadAttention(
        (k_weights): Linear(in_features=768, out_features=768, bias=True)
        (q_weights): Linear(in_features=768, out_features=768, bias=True)
        (v_weights): Linear(in_features=768, out_features=768, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): DummyLayerNorm()
      (norm2): DummyLayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): DummyTransformerBlock(
      (attention): MultiheadAtte

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def generate_text_simple(model, idx,max_new_tokens, context_size):
    # print(f'idx from simple :{idx}')
    for _ in range(max_new_tokens):
        idx_cond = idx[: , -context_size:]
        # print(f'idx_cond from simple :{idx_cond}')
        
        with torch.no_grad():
            logits = model(idx_cond).to(device)
            
        # print(f'logit before from simple :{logits.shape}')
        
        logits  = logits[: , -1 ,:]
        # print(f'logit afterfrom simple :{logits.shape}')
        
    
        probs = torch.softmax(logits , dim =-1)
        idx_next = torch.argmax(probs , dim=-1 , keepdim=True)
        idx = torch.cat((idx, idx_next), dim =1).to(device)
    
    return idx


In [20]:
def text_to_token_ids(text,tokenizer):
    encode = tokenizer.encode(text , allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encode).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids , tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [21]:
text_2 = "Every effort moves you"
text_2 = (
"Is the following text 'spam'? Answer with 'yes' or 'no':"
" 'You are a winner you have been specially"
" selected to receive $1000 cash or a $2000 award.'"
)
token_ids = generate_text_simple(
model=model,
idx=text_to_token_ids(text_2, tokenizer),
max_new_tokens=23,
context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

Is the following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'

The following text 'spam'? Answer with 'yes' or 'no': 'You are a winner


In [22]:
print(model)

DummyGPTModel(
  (token_embed): Embedding(50257, 768)
  (pos_embed): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_block): Sequential(
    (0): DummyTransformerBlock(
      (attention): MultiheadAttention(
        (k_weights): Linear(in_features=768, out_features=768, bias=True)
        (q_weights): Linear(in_features=768, out_features=768, bias=True)
        (v_weights): Linear(in_features=768, out_features=768, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): DummyLayerNorm()
      (norm2): DummyLayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): DummyTransformerBlock(
      (attention): MultiheadAtte

In [23]:
for param in model.parameters():
    param.requires_grad = False
    

In [24]:
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(
in_features=BASE_CONFIG["emb_dim"],
out_features=num_classes
)

In [25]:
for param in model.trf_block[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True
    


In [26]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape)
with torch.no_grad():
    output  = model(inputs)
print(f' output :{output}')

Inputs: tensor([[5211,  345,  423,  640]])
Inputs dimensions: torch.Size([1, 4])
 output :tensor([[[-1.5854,  0.9904],
         [-3.7235,  7.4548],
         [-2.2661,  6.6049],
         [-3.5983,  3.9902]]])


In [27]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
    loss = torch.nn.functional.cross_entropy(logits , target_batch )
    return loss

In [28]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")

    elif num_batches is None:

        num_batches = len(data_loader)

    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(
            input_batch, target_batch, model, device
            )
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [29]:
with torch.no_grad():

    train_loss = calc_loss_loader(
    train_loader, model, device, num_batches=5
    )
val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)
print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

Training loss: 2.183
Validation loss: 2.747
Test loss: 1.899


In [30]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
        train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
        val_loader, model, device, num_batches=eval_iter
        )
    model.train()
    return train_loss, val_loss

In [31]:
def train_classifier_simple(model, train_loader, val_loader, 
                            optimizer, device,
                            num_epochs, eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1
    
    for epoch in num_epochs:
        model.train()
        
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch ,device)
            
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1
        
            if global_step % eval_freq == 0:

                    train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                    print(f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {train_loss:.3f}, "
                    f"Val loss {val_loss:.3f}"
                    )
        return train_losses, val_losses, train_accs, val_accs, examples_seen
    

In [32]:
import time
start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = \
train_classifier_simple(
model, train_loader, val_loader, optimizer, device,
num_epochs=num_epochs, eval_freq=50,
eval_iter=5
)
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

: 